In [ ]:
import pandas as pd
import numpy as np

print("=" * 50)
print("SPENDDNA")
print("YOUR WALLET'S YEAR-END STORY")
print("=" * 50)

print("\nEnvironment setup successful!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

SPENDDNA
YOUR WALLET'S YEAR-END STORY

Environment setup successful!
Pandas version: 2.2.3
NumPy version: 2.4.6


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving rahul_transactions[1].csv to rahul_transactions[1].csv


In [ ]:
df = pd.read_csv("rahul_transactions.csv")

print("DATASET LOADED SUCCESSFULLY!")
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nFIRST 5 TRANSACTIONS:")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'rahul_transactions.csv'

In [ ]:
import os

print("FILES AVAILABLE IN COLAB:")
print(os.listdir())

FILES AVAILABLE IN COLAB:
['.config', '21075970-DADS_September_Minor_Project_02.zip', 'rahul_transactions[1].csv', 'sample_data']


In [ ]:
df = pd.read_csv("rahul_transactions[1].csv")

print("DATASET LOADED SUCCESSFULLY!")
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nFIRST 5 TRANSACTIONS:")
display(df.head())



DATASET LOADED SUCCESSFULLY!
Total rows: 1328
Total columns: 8

COLUMN NAMES:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

FIRST 5 TRANSACTIONS:


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [ ]:
print("SPENDDNA - DATASET QUALITY REPORT")
print("=" * 45)

print("\nDATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES:")
print(df.isnull().sum())

print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

print("\nTRANSACTION TYPES:")
print(df["Type"].value_counts())

print("\nUNIQUE TRANSACTION DESCRIPTIONS:")
print(df["Description"].nunique())

SPENDDNA - DATASET QUALITY REPORT

DATA TYPES:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object

MISSING VALUES:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64

DUPLICATE ROWS:
18

TRANSACTION TYPES:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64

UNIQUE TRANSACTION DESCRIPTIONS:
283


In [ ]:
# SPENDDNA - FEATURE 1: TRANSACTION PARSER

# Create a separate copy
df_clean = df.copy()

# 1. Standardise dates
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# 2. Clean transaction amounts
df_clean["Amount"] = (
    df_clean["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean["Amount"] = pd.to_numeric(
    df_clean["Amount"],
    errors="coerce"
)

# 3. Standardise transaction types
df_clean["Type"] = (
    df_clean["Type"]
    .str.strip()
    .str.upper()
    .replace({
        "DR": "debit",
        "DEBIT": "debit",
        "CR": "credit",
        "CREDIT": "credit"
    })
)

# 4. Remove duplicate transactions
original_count = len(df_clean)

df_clean = df_clean.drop_duplicates()

duplicates_removed = original_count - len(df_clean)

# 5. Remove invalid dates and amounts
invalid_dates = df_clean["Date"].isna().sum()
invalid_amounts = df_clean["Amount"].isna().sum()

df_clean = df_clean.dropna(
    subset=["Date", "Amount"]
)

# 6. Display cleaning report
print("SPENDDNA - TRANSACTION PARSER")
print("=" * 40)

print("Original transactions:", original_count)
print("Duplicates removed:", duplicates_removed)
print("Invalid dates:", invalid_dates)
print("Invalid amounts:", invalid_amounts)
print("Clean transactions:", len(df_clean))

print("\nSTANDARDISED TRANSACTION TYPES:")
print(df_clean["Type"].value_counts())

print("\nCLEANED DATA TYPES:")
print(df_clean.dtypes)

SPENDDNA - TRANSACTION PARSER
Original transactions: 1328
Duplicates removed: 18
Invalid dates: 0
Invalid amounts: 0
Clean transactions: 1310

STANDARDISED TRANSACTION TYPES:
Type
debit     1304
credit       6
Name: count, dtype: int64

CLEANED DATA TYPES:
Date           datetime64[ns]
Time                   object
Description            object
Type                   object
Amount                float64
Balance               float64
Mode                   object
Ref                    object
dtype: object


In [19]:
# SPENDDNA - FEATURE 2: VENDOR EXTRACTOR

def extract_vendor(description):

    text = str(description).upper().strip()

    # Food delivery
    if "SWIGGY" in text or "BUNDL" in text or "INSTAMART" in text:
        return "Swiggy / Instamart"
    elif "ZOMATO" in text:
        return "Zomato"

    # Quick commerce and groceries
    elif "BLINKIT" in text or "GROFERS" in text:
        return "Blinkit"
    elif "ZEPTO" in text:
        return "Zepto"
    elif "BIGBASKET" in text:
        return "BigBasket"
    elif "DMART" in text or "AVENUE SUPERMARTS" in text:
        return "DMart"
    elif "KIRANAKART" in text:
        return "KiranaKart"

    # E-commerce and shopping
    elif "AMAZON PRIME VIDEO" in text:
        return "Amazon Prime Video"
    elif "AMAZON PRIME" in text or "AMAZON-PRIME" in text or "AMZN PRIME" in text:
        return "Amazon Prime"
    elif "AMAZON" in text or "AMZN" in text:
        return "Amazon"
    elif "FLIPKART" in text or "FKART" in text:
        return "Flipkart"
    elif "MYNTRA" in text:
        return "Myntra"
    elif "NYKAA" in text or "FSN E-COMMERCE" in text:
        return "Nykaa"
    elif "INNOVATIVE RETAIL" in text:
        return "Innovative Retail"

    # Transport
    elif "OLA ELECTRIC" in text:
        return "Ola Electric"
    elif "OLA" in text or "ANI TECHNOLOGIES" in text:
        return "Ola"
    elif "UBER" in text:
        return "Uber"
    elif "RAPIDO" in text or "ROPPEN" in text:
        return "Rapido"
    elif "BMTC" in text or "TUMMOC" in text:
        return "BMTC"

    # Restaurants and cafes
    elif "STARBUCKS" in text:
        return "Starbucks"
    elif "THIRD WAVE" in text or "TWC INDIA" in text:
        return "Third Wave Coffee"
    elif "COFFEE DAY" in text or "CCD" in text:
        return "Cafe Coffee Day"
    elif "EMPIRE RESTAURANT" in text:
        return "Empire Restaurant"
    elif "MEGHANA" in text:
        return "Meghana Foods"
    elif "TRUFFLES" in text:
        return "Truffles"
    elif "DINEOUT" in text:
        return "Dineout"
    elif "RESTAURANT" in text:
        return "Other Restaurant"

    # Entertainment and subscriptions
    elif "BOOKMYSHOW" in text or "BMS" in text or "BIGTREE" in text:
        return "BookMyShow"
    elif "HOTSTAR" in text or "DISNEY" in text or "STAR INDIA" in text:
        return "Disney+ Hotstar"
    elif "NETFLIX" in text:
        return "Netflix"
    elif "SPOTIFY" in text:
        return "Spotify"

    # Utilities
    elif "BESCOM" in text or "BANGALORE ELEC" in text:
        return "BESCOM"
    elif "BWSSB" in text:
        return "BWSSB"
    elif "AIRTEL" in text:
        return "Airtel"
    elif "JIOFIBER" in text:
        return "JioFiber"
    elif "JIO" in text:
        return "Jio"
    elif "VODAFONE" in text or text.startswith("VI"):
        return "Vi"

    # Fuel
    elif "HP PETROL" in text:
        return "HP Petrol"
    elif "INDIAN OIL" in text or "IOC" in text:
        return "Indian Oil"
    elif "BPCL" in text:
        return "BPCL"

    # Investments
    elif "ZERODHA" in text:
        return "Zerodha"
    elif "GROWW" in text:
        return "Groww"

    # Other transactions
    elif "ATM-WDL" in text:
        return "ATM Withdrawal"
    elif "RENT-LANDLORD" in text:
        return "Rent"
    elif "TECHCRUSH" in text:
        return "TechCrush Labs"

    # Personal transfers
    elif text.startswith("UPI-") or text.startswith("UPI "):
        return "Personal Transfer"

    return "Other / Unidentified"


print("SpendDNA Vendor Extractor created successfully!")

SpendDNA Vendor Extractor created successfully!


In [17]:
# TEST VENDOR EXTRACTION

test_descriptions = [
    "POS SWIGGY BANGALORE",
    "ANI Technologies",
    "AMAZON PRIME VIDEO",
    "UPI-AMAZON-PRIME@HDFCBANK",
    "BHIM-BLINKIT",
    "POS EMPIRE RESTAURANT",
    "UPI-UBER-2426@HDFCBANK",
    "ATM-WDL-HDFC-3609",
    "UPI-AMAN-8934@OKAXIS"
]

for item in test_descriptions:
    print(item, "---->", extract_vendor(item))

POS SWIGGY BANGALORE ----> Swiggy / Instamart
ANI Technologies ----> Ola
AMAZON PRIME VIDEO ----> Amazon Prime Video
UPI-AMAZON-PRIME@HDFCBANK ----> Amazon Prime
BHIM-BLINKIT ----> Blinkit
POS EMPIRE RESTAURANT ----> Empire Restaurant
UPI-UBER-2426@HDFCBANK ----> Uber
ATM-WDL-HDFC-3609 ----> ATM Withdrawal
UPI-AMAN-8934@OKAXIS ----> Personal Transfer


In [18]:
# APPLY VENDOR EXTRACTOR TO FULL DATASET

df_clean["Vendor"] = df_clean["Description"].apply(extract_vendor)

print("VENDOR EXTRACTION COMPLETED!")
print("=" * 40)

print("Total transactions:", len(df_clean))
print("Unique vendors:", df_clean["Vendor"].nunique())

print("\nTOP 20 VENDORS:")
print(df_clean["Vendor"].value_counts().head(20))

print("\nUNIDENTIFIED TRANSACTIONS:")
print(
    df_clean[df_clean["Vendor"] == "Other / Unidentified"]
    ["Description"].value_counts()
)

VENDOR EXTRACTION COMPLETED!
Total transactions: 1310
Unique vendors: 46

TOP 20 VENDORS:
Vendor
Swiggy / Instamart    243
Zomato                121
Amazon                 76
Uber                   71
Ola                    67
Zepto                  58
Rapido                 55
Blinkit                55
Flipkart               47
Starbucks              42
BMTC                   37
Personal Transfer      32
Other Restaurant       29
Cafe Coffee Day        26
DMart                  22
Ola Electric           20
Third Wave Coffee      20
Myntra                 20
Nykaa                  19
Indian Oil             17
Name: count, dtype: int64

UNIDENTIFIED TRANSACTIONS:
Description
INNOVATIVE RETAIL    8
Name: count, dtype: int64


In [20]:
df_clean["Vendor"] = df_clean["Description"].apply(extract_vendor)

print("UPDATED VENDOR COUNT:", df_clean["Vendor"].nunique())

print("\nUNIDENTIFIED TRANSACTIONS:")
print(
    df_clean[df_clean["Vendor"] == "Other / Unidentified"]
    ["Description"].value_counts()
)

UPDATED VENDOR COUNT: 46

UNIDENTIFIED TRANSACTIONS:
Series([], Name: count, dtype: int64)


In [23]:
# SPENDDNA - FEATURE 3: CATEGORY TAGGER

def categorize_transaction(row):

    vendor = row["Vendor"]
    description = str(row["Description"]).upper()

    # Income
    if row["Type"] == "credit":
        return "Income"

    # Cash withdrawal
    if vendor == "ATM Withdrawal":
        return "Cash Withdrawal"

    # Personal transfers
    if vendor == "Personal Transfer":
        return "Personal Transfer"
    # Rent payments
    if vendor == "Rent":
        return "Rent"

    # Food delivery and quick commerce
    if vendor == "Swiggy / Instamart":
        if "INSTAMART" in description or "BUNDL" in description:
            return "Quick Commerce"
        return "Food Delivery"

    if vendor == "Zomato":
        if "DINING" in description:
            return "Restaurants"
        return "Food Delivery"

    # Shopping
    if vendor in ["Amazon", "Flipkart", "Myntra", "Nykaa",
                  "Innovative Retail"]:
        return "E-commerce"

    # Quick commerce
    if vendor in ["Blinkit", "Zepto", "KiranaKart"]:
        return "Quick Commerce"

    # Groceries
    if vendor in ["BigBasket", "DMart"]:
        return "Groceries"

    # Transport
    if vendor in ["Uber", "Ola", "Ola Electric", "Rapido", "BMTC"]:
        return "Transport"

    # Cafes
    if vendor in ["Starbucks", "Third Wave Coffee", "Cafe Coffee Day"]:
        return "Cafe"

    # Restaurants
    if vendor in ["Empire Restaurant", "Meghana Foods",
                  "Truffles", "Dineout", "Other Restaurant"]:
        return "Restaurants"

    # Subscriptions
    if vendor in ["Netflix", "Spotify", "Disney+ Hotstar",
                  "Amazon Prime", "Amazon Prime Video"]:
        return "Subscriptions"

    # Entertainment
    if vendor == "BookMyShow":
        return "Entertainment"

    # Utilities
    if vendor in ["BESCOM", "BWSSB", "Airtel", "Jio",
                  "JioFiber", "Vi"]:
        return "Utilities"

    # Investments
    if vendor in ["Zerodha", "Groww"]:
        return "Investments"

    # Fuel
    if vendor in ["HP Petrol", "Indian Oil", "BPCL"]:
        return "Fuel"

    return "Other"


print("SpendDNA Category Tagger created successfully!")

SpendDNA Category Tagger created successfully!


In [24]:
# APPLY CATEGORY TAGGER

df_clean["Category"] = df_clean.apply(
    categorize_transaction,
    axis=1
)

print("CATEGORY TAGGING COMPLETED!")
print("=" * 40)

print("Total transactions:", len(df_clean))
print("Unique categories:", df_clean["Category"].nunique())

print("\nTRANSACTIONS BY CATEGORY:")
print(df_clean["Category"].value_counts())

print("\nUNCATEGORISED TRANSACTIONS:")
print(df_clean[df_clean["Category"] == "Other"]["Vendor"].value_counts())

CATEGORY TAGGING COMPLETED!
Total transactions: 1310
Unique categories: 16

TRANSACTIONS BY CATEGORY:
Category
Food Delivery        259
Transport            250
Quick Commerce       211
E-commerce           170
Restaurants           93
Cafe                  88
Subscriptions         41
Utilities             40
Groceries             33
Personal Transfer     32
Fuel                  28
Investments           23
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64

UNCATEGORISED TRANSACTIONS:
Series([], Name: count, dtype: int64)


In [25]:
# UPDATE ALL TRANSACTION CATEGORIES

df_clean["Category"] = df_clean.apply(
    categorize_transaction,
    axis=1
)

print("UPDATED CATEGORY REPORT")
print("=" * 40)

print(df_clean["Category"].value_counts())

print("\nOTHER CATEGORY CHECK:")
print(df_clean[df_clean["Category"] == "Other"]["Vendor"].value_counts())

UPDATED CATEGORY REPORT
Category
Food Delivery        259
Transport            250
Quick Commerce       211
E-commerce           170
Restaurants           93
Cafe                  88
Subscriptions         41
Utilities             40
Groceries             33
Personal Transfer     32
Fuel                  28
Investments           23
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64

OTHER CATEGORY CHECK:
Series([], Name: count, dtype: int64)


In [26]:
# SPENDDNA - FEATURE 4: SPENDING OVERVIEW

# Separate debit and credit transactions
debit_df = df_clean[df_clean["Type"] == "debit"]
credit_df = df_clean[df_clean["Type"] == "credit"]

# Financial calculations
total_spent = debit_df["Amount"].sum()
total_income = credit_df["Amount"].sum()

net_savings = total_income - total_spent

if total_income > 0:
    savings_rate = (net_savings / total_income) * 100
else:
    savings_rate = 0

# Transaction statistics
total_transactions = len(df_clean)
debit_count = len(debit_df)
credit_count = len(credit_df)

# Top 5 spending categories
top_categories = (
    debit_df.groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

# Top 5 vendors
top_vendors = (
    debit_df.groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

# Display financial overview
print("=" * 50)
print("SPENDDNA - YOUR FINANCIAL OVERVIEW")
print("=" * 50)

print(f"\nTotal Transactions: {total_transactions}")
print(f"Debit Transactions: {debit_count}")
print(f"Credit Transactions: {credit_count}")

print(f"\nTotal Money Spent: ₹{total_spent:,.2f}")
print(f"Total Income: ₹{total_income:,.2f}")
print(f"Net Savings: ₹{net_savings:,.2f}")
print(f"Savings Rate: {savings_rate:.2f}%")

print("\nTOP 5 SPENDING CATEGORIES:")
print(top_categories)

print("\nTOP 5 SPENDING VENDORS:")
print(top_vendors)

SPENDDNA - YOUR FINANCIAL OVERVIEW

Total Transactions: 1310
Debit Transactions: 1304
Credit Transactions: 6

Total Money Spent: ₹1,678,901.00
Total Income: ₹509,774.00
Net Savings: ₹-1,169,127.00
Savings Rate: -229.34%

TOP 5 SPENDING CATEGORIES:
Category
E-commerce       602968.0
Investments      248160.0
Restaurants      127290.0
Food Delivery    112275.0
Rent             108000.0
Name: Amount, dtype: float64

TOP 5 SPENDING VENDORS:
Vendor
Amazon                318422.0
Zerodha               210000.0
Flipkart              177510.0
Rent                  108000.0
Swiggy / Instamart    104351.0
Name: Amount, dtype: float64


In [27]:
# SPENDDNA - FEATURE 5: MONTHLY SPENDING TRENDS

# Select only debit transactions
monthly_df = df_clean[df_clean["Type"] == "debit"].copy()

# Extract month from transaction date
monthly_df["Month"] = monthly_df["Date"].dt.strftime("%b %Y")

# Category-wise monthly spending matrix
monthly_category = (
    monthly_df.groupby(["Category", "Month"])["Amount"]
    .sum()
    .unstack(fill_value=0)
)

# Arrange months chronologically
monthly_category = monthly_category.reindex(
    sorted(
        monthly_category.columns,
        key=lambda x: pd.to_datetime(x, format="%b %Y")
    ),
    axis=1
)

# Total spending for each month
monthly_total = monthly_df.groupby("Month")["Amount"].sum()

monthly_total = monthly_total.reindex(monthly_category.columns)

# Month-to-month spending changes
monthly_change = monthly_total.diff()

# Highest and lowest spending months
highest_month = monthly_total.idxmax()
lowest_month = monthly_total.idxmin()

print("=" * 55)
print("SPENDDNA - MONTHLY SPENDING TRENDS")
print("=" * 55)

print("\nCATEGORY-WISE MONTHLY SPENDING:")
display(monthly_category.round(2))

print("\nTOTAL SPENDING BY MONTH:")
print(monthly_total.round(2))

print("\nHIGHEST SPENDING MONTH:")
print(highest_month, "₹", round(monthly_total.max(), 2))

print("\nLOWEST SPENDING MONTH:")
print(lowest_month, "₹", round(monthly_total.min(), 2))

print("\nMONTH-TO-MONTH SPENDING CHANGE:")
print(monthly_change.round(2))

print("\nBIGGEST SPENDING INCREASE:")
print(monthly_change.idxmax(), "₹", round(monthly_change.max(), 2))

print("\nBIGGEST SPENDING DECREASE:")
print(monthly_change.idxmin(), "₹", round(monthly_change.min(), 2))

SPENDDNA - MONTHLY SPENDING TRENDS

CATEGORY-WISE MONTHLY SPENDING:


Month,Jan 2024,Feb 2024,Mar 2024,Apr 2024,May 2024,Jun 2024
Category,,,,,,
Cafe,2824.0,4044.0,4521.0,5826.0,5668.0,5056.0
Cash Withdrawal,2000.0,5000.0,8000.0,5500.0,8000.0,17000.0
E-commerce,97134.0,93827.0,104538.0,73621.0,96857.0,136991.0
Entertainment,1263.0,474.0,2418.0,2224.0,0.0,1914.0
Food Delivery,19968.0,16431.0,16844.0,20392.0,20938.0,17702.0
Fuel,30322.0,2079.0,26164.0,18718.0,9138.0,2882.0
Groceries,17649.0,7517.0,5523.0,6225.0,7862.0,5432.0
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0
Personal Transfer,8718.0,4514.0,5552.0,2055.0,3412.0,5586.0



TOTAL SPENDING BY MONTH:
Month
Jan 2024    290767.0
Feb 2024    234071.0
Mar 2024    330458.0
Apr 2024    251382.0
May 2024    277306.0
Jun 2024    294917.0
Name: Amount, dtype: float64

HIGHEST SPENDING MONTH:
Mar 2024 ₹ 330458.0

LOWEST SPENDING MONTH:
Feb 2024 ₹ 234071.0

MONTH-TO-MONTH SPENDING CHANGE:
Month
Jan 2024        NaN
Feb 2024   -56696.0
Mar 2024    96387.0
Apr 2024   -79076.0
May 2024    25924.0
Jun 2024    17611.0
Name: Amount, dtype: float64

BIGGEST SPENDING INCREASE:
Mar 2024 ₹ 96387.0

BIGGEST SPENDING DECREASE:
Apr 2024 ₹ -79076.0


In [28]:
# SPENDDNA - FEATURE 6: TIME-OF-DAY ANALYSIS

# Select debit transactions
time_df = df_clean[df_clean["Type"] == "debit"].copy()

# Extract hour from transaction time
time_df["Hour"] = pd.to_numeric(
    time_df["Time"].astype(str).str.split(":").str[0],
    errors="coerce"
)

# Remove invalid hours
time_df = time_df.dropna(subset=["Hour"])
time_df["Hour"] = time_df["Hour"].astype(int)

# Category-wise spending by hour
hourly_category = (
    time_df.groupby(["Category", "Hour"])["Amount"]
    .sum()
    .unstack(fill_value=0)
)

# Total spending for each hour
hourly_total = time_df.groupby("Hour")["Amount"].sum()

# Highest spending hour
peak_hour = hourly_total.idxmax()

print("=" * 55)
print("SPENDDNA - TIME-OF-DAY ANALYSIS")
print("=" * 55)

print("\nCATEGORY-WISE SPENDING BY HOUR:")
display(hourly_category.round(2))

print("\nTOTAL SPENDING BY HOUR:")
print(hourly_total.round(2))

print("\nHIGHEST SPENDING HOUR:")
print(f"{peak_hour}:00 - ₹{hourly_total.max():,.2f}")

print("\nTOP 5 SPENDING HOURS:")
print(hourly_total.sort_values(ascending=False).head(5))

SPENDDNA - TIME-OF-DAY ANALYSIS

CATEGORY-WISE SPENDING BY HOUR:


Hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
Category,,,,,,,,,,,,,,,,,,,,,
Cafe,872.0,549.0,0.0,316.0,0.0,0.0,179.0,480.0,2668.0,1986.0,...,1083.0,1056.0,3079.0,3114.0,2236.0,372.0,1214.0,0.0,0.0,764.0
Cash Withdrawal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4000.0,10500.0,...,1000.0,0.0,5000.0,0.0,5000.0,0.0,1000.0,1000.0,4000.0,0.0
E-commerce,18650.0,9801.0,10246.0,16538.0,9738.0,13305.0,8423.0,10916.0,6537.0,40581.0,...,74166.0,7700.0,52046.0,35378.0,35272.0,12668.0,51190.0,35095.0,16182.0,14818.0
Entertainment,562.0,620.0,0.0,0.0,0.0,0.0,0.0,311.0,1463.0,0.0,...,0.0,0.0,1881.0,0.0,1270.0,417.0,1426.0,0.0,343.0,0.0
Food Delivery,438.0,2418.0,928.0,1702.0,2133.0,3488.0,238.0,232.0,2870.0,3924.0,...,3834.0,6458.0,4622.0,3667.0,11799.0,11172.0,13160.0,5555.0,6676.0,4437.0
Fuel,2675.0,2048.0,0.0,676.0,2010.0,2105.0,0.0,0.0,1727.0,2283.0,...,7828.0,11937.0,9343.0,0.0,1950.0,8351.0,705.0,11845.0,0.0,1641.0
Groceries,3895.0,1536.0,2352.0,7080.0,1169.0,0.0,5265.0,0.0,2430.0,4684.0,...,655.0,0.0,0.0,0.0,910.0,1248.0,2566.0,3688.0,0.0,2536.0
Investments,4883.0,15000.0,0.0,0.0,18834.0,4496.0,37717.0,0.0,0.0,4737.0,...,0.0,0.0,0.0,0.0,15000.0,4476.0,15000.0,0.0,15000.0,18628.0
Personal Transfer,0.0,0.0,0.0,0.0,654.0,0.0,0.0,0.0,329.0,2425.0,...,2077.0,1029.0,2361.0,4024.0,1540.0,1507.0,2825.0,3004.0,2185.0,0.0



TOTAL SPENDING BY HOUR:
Hour
0      37969.0
1      38882.0
2      20283.0
3      34868.0
4      43901.0
5      33671.0
6      55506.0
7      19938.0
8      41399.0
9      86701.0
10    140526.0
11    123389.0
12     77096.0
13    113819.0
14    121010.0
15     39647.0
16    100597.0
17     56604.0
18    101343.0
19     67848.0
20    138089.0
21     81545.0
22     55867.0
23     48403.0
Name: Amount, dtype: float64

HIGHEST SPENDING HOUR:
10:00 - ₹140,526.00

TOP 5 SPENDING HOURS:
Hour
10    140526.0
20    138089.0
11    123389.0
14    121010.0
13    113819.0
Name: Amount, dtype: float64


In [29]:
# SPENDDNA - FEATURE 7: ANOMALY DETECTION

# Select debit transactions
anomaly_df = df_clean[
    df_clean["Type"] == "debit"
].copy()

# Calculate category-wise average and standard deviation
category_stats = anomaly_df.groupby("Category")["Amount"].agg(
    ["mean", "std"]
)

# Attach category statistics to each transaction
anomaly_df["Category_Average"] = anomaly_df["Category"].map(
    category_stats["mean"]
)

anomaly_df["Category_Std"] = anomaly_df["Category"].map(
    category_stats["std"]
)

# Calculate Z-score safely
anomaly_df["Z_Score"] = np.where(
    anomaly_df["Category_Std"] > 0,
    (anomaly_df["Amount"] - anomaly_df["Category_Average"])
    / anomaly_df["Category_Std"],
    0
)

# Flag unusually high transactions
anomaly_df["Anomaly"] = anomaly_df["Z_Score"] > 2

# Extract flagged transactions
anomalies = anomaly_df[
    anomaly_df["Anomaly"]
].sort_values("Z_Score", ascending=False)

print("=" * 55)
print("SPENDDNA - ANOMALY DETECTION REPORT")
print("=" * 55)

print("\nTotal debit transactions:", len(anomaly_df))
print("Unusual transactions detected:", len(anomalies))

print("\nTOP UNUSUAL TRANSACTIONS:")

display(
    anomalies[
        ["Date", "Description", "Category", "Amount",
         "Category_Average", "Z_Score"]
    ].head(15).round(2)
)

print("\nANOMALIES BY CATEGORY:")
print(anomalies["Category"].value_counts())

SPENDDNA - ANOMALY DETECTION REPORT

Total debit transactions: 1304
Unusual transactions detected: 24

TOP UNUSUAL TRANSACTIONS:


,Date,Description,Category,Amount,Category_Average,Z_Score
414,2024-02-26,POS BANGALORE RESTAURANT,Restaurants,8383.0,1368.71,4.35
1271,2024-06-22,POS DINEOUT,Restaurants,7935.0,1368.71,4.07
1298,2024-06-26,AMAZONIN MARKETPLACE,E-commerce,22008.0,3546.87,4.07
653,2024-03-31,POS MEGHANA FOODS,Restaurants,7931.0,1368.71,4.07
269,2024-02-07,UPI-AMAZONPAY@HDFCBANK,E-commerce,21986.0,3546.87,4.06
464,2024-03-04,POS TRUFFLES,Restaurants,7441.0,1368.71,3.76
49,2024-01-07,POS BANGALORE RESTAURANT,Restaurants,7314.0,1368.71,3.69
475,2024-03-05,UPI-AMAZONPAY@HDFCBANK,E-commerce,19917.0,3546.87,3.61
450,2024-03-02,AMAZON IN,E-commerce,18273.0,3546.87,3.25
1079,2024-05-27,UPI-FLIPKART@HDFCBANK,E-commerce,17831.0,3546.87,3.15



ANOMALIES BY CATEGORY:
Category
E-commerce           15
Restaurants           5
Fuel                  2
Personal Transfer     1
Quick Commerce        1
Name: count, dtype: int64


In [30]:
# SPENDDNA - FEATURE 8: SPENDING PERSONALITY

# Use debit transactions only
personality_df = df_clean[
    df_clean["Type"] == "debit"
].copy()

total_spending = personality_df["Amount"].sum()

# Calculate category-wise spending percentages
category_spending = personality_df.groupby(
    "Category"
)["Amount"].sum()

category_percentage = (
    category_spending / total_spending
) * 100

# Calculate late-night food delivery behaviour
food_df = personality_df[
    personality_df["Category"] == "Food Delivery"
].copy()

food_df["Hour"] = pd.to_numeric(
    food_df["Time"].astype(str).str.split(":").str[0],
    errors="coerce"
)

late_night_food = food_df[
    (food_df["Hour"] >= 21) | (food_df["Hour"] <= 2)
]

if len(food_df) > 0:
    late_night_percentage = (
        len(late_night_food) / len(food_df)
    ) * 100
else:
    late_night_percentage = 0

# Count distinct subscription vendors
subscription_vendors = personality_df[
    personality_df["Category"] == "Subscriptions"
]["Vendor"].nunique()

# Calculate savings rate
total_income = df_clean[
    df_clean["Type"] == "credit"
]["Amount"].sum()

net_savings = total_income - total_spending

if total_income > 0:
    savings_rate = (net_savings / total_income) * 100
else:
    savings_rate = 0

# Detect spending personalities
archetypes = []

if sum(category_percentage.get(c, 0) for c in
       ["Food Delivery", "Restaurants", "Cafe"]) > 25:
    archetypes.append("The Foodie")

if category_percentage.get("Quick Commerce", 0) > 15:
    archetypes.append("Quick Commerce Junkie")

if category_percentage.get("E-commerce", 0) > 15:
    archetypes.append("The Shopaholic")

if category_percentage.get("Investments", 0) > 15:
    archetypes.append("The Investor")

if late_night_percentage > 50:
    archetypes.append("Late-Night Snacker")

if category_percentage.get("Transport", 0) > 10:
    archetypes.append("Cab Commuter")

if subscription_vendors >= 5:
    archetypes.append("Subscription Lover")

if savings_rate < 10:
    archetypes.append("YOLO Spender")

if savings_rate > 40:
    archetypes.append("Disciplined Saver")

# Display results
print("=" * 55)
print("SPENDDNA - YOUR SPENDING PERSONALITY")
print("=" * 55)

print("\nCATEGORY SPENDING PERCENTAGES:")
print(category_percentage.round(2))

print("\nLATE-NIGHT FOOD DELIVERY:")
print(f"{late_night_percentage:.2f}%")

print("\nDISTINCT SUBSCRIPTION VENDORS:")
print(subscription_vendors)

print("\nSAVINGS RATE:")
print(f"{savings_rate:.2f}%")

print("\nYOUR SPENDING ARCHETYPES:")

if archetypes:
    for personality in archetypes:
        print("★", personality)
else:
    print("No specific archetype detected.")

print("\nTOTAL ARCHETYPES DETECTED:", len(archetypes))

SPENDDNA - YOUR SPENDING PERSONALITY

CATEGORY SPENDING PERCENTAGES:
Category
Cafe                  1.66
Cash Withdrawal       2.71
E-commerce           35.91
Entertainment         0.49
Food Delivery         6.69
Fuel                  5.32
Groceries             2.99
Investments          14.78
Personal Transfer     1.78
Quick Commerce        6.13
Rent                  6.43
Restaurants           7.58
Subscriptions         1.70
Transport             3.42
Utilities             2.39
Name: Amount, dtype: float64

LATE-NIGHT FOOD DELIVERY:
19.69%

DISTINCT SUBSCRIPTION VENDORS:
5

SAVINGS RATE:
-229.34%

YOUR SPENDING ARCHETYPES:
★ The Shopaholic
★ Subscription Lover
★ YOLO Spender

TOTAL ARCHETYPES DETECTED: 3


In [31]:
# SPENDDNA - FINAL PROJECT VALIDATION

print("=" * 50)
print("SPENDDNA - FINAL VALIDATION")
print("=" * 50)

print("Original transactions:", len(df))
print("Clean transactions:", len(df_clean))
print("Unique vendors:", df_clean["Vendor"].nunique())
print("Unique categories:", df_clean["Category"].nunique())
print("Unidentified vendors:",
      (df_clean["Vendor"] == "Other / Unidentified").sum())
print("Uncategorised transactions:",
      (df_clean["Category"] == "Other").sum())
print("Anomalies detected:", len(anomalies))
print("Archetypes detected:", len(archetypes))

print("\nAVAILABLE FEATURES:")
print(df_clean.columns.tolist())

SPENDDNA - FINAL VALIDATION
Original transactions: 1328
Clean transactions: 1310
Unique vendors: 46
Unique categories: 16
Unidentified vendors: 0
Uncategorised transactions: 0
Anomalies detected: 24
Archetypes detected: 3

AVAILABLE FEATURES:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref', 'Vendor', 'Category']
